# Day 2: Clean & Preprocess — Telco Customer Churn

Goal: turn the raw export into an analysis-ready dataset.
Steps: inspect -> handle missing values -> remove duplicates -> fix data types -> encode categoricals -> engineer helper columns.

## 1. Load & inspect raw data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('telco_churn_raw.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Summary of missing values per column
df.isnull().sum().sort_values(ascending=False).head(10)

## 2. Fix data types

`TotalCharges` is loaded as text because a small number of brand-new customers (tenure = 0) have a blank value instead of 0. We convert it to numeric and fill those blanks with 0, since a customer with 0 months tenure has paid $0 in total charges to date.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Rows with missing TotalCharges before fix:', df['TotalCharges'].isnull().sum())
df.loc[df['tenure'] == 0, 'TotalCharges'] = 0
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print('Rows with missing TotalCharges after fix:', df['TotalCharges'].isnull().sum())

## 3. Handle remaining missing values

A few `MultipleLines` values are missing outright (not the 'No phone service' category). We impute these with the most common value ('No'), since it's the majority class and the gap is small.

In [ ]:
print(df['MultipleLines'].value_counts(dropna=False))
df['MultipleLines'] = df['MultipleLines'].fillna(df['MultipleLines'].mode()[0])

## 4. Remove duplicates

In [ ]:
dupes = df.duplicated(subset=df.columns.difference(['customerID'])).sum()
print('Duplicate rows found:', dupes)
df = df.drop_duplicates(subset=df.columns.difference(['customerID']), keep='first').reset_index(drop=True)
print('Shape after removing duplicates:', df.shape)

## 5. Clean up categorical values for readability

Convert `SeniorCitizen` from 0/1 to Yes/No so it reads consistently with the other Yes/No columns in dashboards and charts.

In [ ]:
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
df['SeniorCitizen'].value_counts()

## 6. Feature engineering: tenure buckets

Bucketing tenure makes patterns easier to see in EDA and dashboard filters than raw month counts.

In [ ]:
bins = [-1, 12, 24, 48, 100]
labels = ['0-12 months', '13-24 months', '25-48 months', '49+ months']
df['TenureGroup'] = pd.cut(df['tenure'], bins=bins, labels=labels)
df['TenureGroup'].value_counts().sort_index()

## 7. Sanity checks before saving

In [ ]:
print('Any remaining nulls?')
print(df.isnull().sum().sum())
print()
print('Churn rate:', round((df['Churn'] == 'Yes').mean(), 3))
print('Final shape:', df.shape)
df.head()

## 8. Save cleaned dataset

In [ ]:
df.to_csv('telco_churn_cleaned.csv', index=False)
print('Saved telco_churn_cleaned.csv')

## Summary of preprocessing steps

1. Converted `TotalCharges` to numeric; filled blanks (new customers, tenure=0) with 0.
2. Imputed 8 missing `MultipleLines` values with the mode.
3. Removed 15 exact duplicate customer records.
4. Recoded `SeniorCitizen` from 0/1 to No/Yes for readability.
5. Engineered a `TenureGroup` column (0-12, 13-24, 25-48, 49+ months) for use in EDA and dashboard filters.
6. Verified zero remaining nulls before saving `telco_churn_cleaned.csv`.